In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
df = spark.read\
    .format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("/Volumes/business_to_business/sports_bar_data/orders/*")\
    .withColumn("ingestion_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_name", "_metadata.file_size")


In [0]:
files = dbutils.fs.ls("/Volumes/business_to_business/sports_bar_data/orders")
print(files)
for file in files:
    dbutils.fs.mv(
        file.path,
        "/Volumes/business_to_business/sports_bar_data/processed_orders/" + file.name
    )
   

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df.write\
.mode("overwrite")\
.format("delta")\
.option("delta.enableChangeDataFeed", "true")\
.option("mergeSchema", "true")\
.saveAsTable("business_to_business.bronze.orders")

In [0]:
silver_df = spark.read.table("business_to_business.bronze.orders")
display(silver_df.limit(5))

### Keep the rows only where row quantity is present

In [0]:
silver_df = silver_df.filter(F.col("order_qty").isNotNull())
silver_df.show()

In [0]:
orders_df = silver_df.withColumn("customer_id", 
                     F.when(F.col("customer_id").cast("string").rlike(r"^[0-9]+$"), F.col("customer_id").cast("string"))
                     .otherwise(F.lit(999999).cast("string"))
                     )

In [0]:
display(orders_df.limit(5))



In [0]:
orders_df = orders_df.withColumn("order_placement_date",
                     F.regexp_replace("order_placement_date", r"^[A-Za-z]+,\s*", "")
                     )

In [0]:
orders_df = orders_df.withColumn("order_placement_date",
                     F.coalesce(
                        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
                        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
                        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
                        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
                     )
                     )

In [0]:
columns  = df.columns
columns = columns[:-3]
columns
# silver_df.count()


### Drop the dublicate rows

In [0]:
orders_df = orders_df.dropDuplicates(columns)
orders_df.count()

In [0]:
display(orders_df.limit(3))

In [0]:
orders_df = (
    orders_df.withColumn("product_id", F.col("product_id").cast("string"))
)

In [0]:
display(orders_df)

In [0]:
# check the maximum and minimum date 
orders_df.agg(F.min("order_placement_date"), F.max("order_placement_date")).show()

In [0]:
products_df = spark.read.table("business_to_business.silver.dim_sbproducts")
# display(products_df)
orders_df = orders_df.join(products_df, "product_id", how="inner").select(
    orders_df["*"]
    ,products_df["product_code"]
)

In [0]:
display(orders_df)

In [0]:
if not (spark.catalog.tableExists("business_to_business.silver.orders")):
    orders_df.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable("business_to_business.silver.orders")
else:
    silver_delta = DeltaTable.forName(spark, "business_to_business.silver.orders")
    silver_delta.alias("silver").merge(orders_df.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Gold Layer

In [0]:
orders_gold_df = spark.read.table("business_to_business.silver.orders")

In [0]:
display(orders_gold_df.limit(3))

In [0]:
df_gold = spark.sql(f'''SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM business_to_business.silver.orders;''')
display(df_gold)

In [0]:
if not (spark.catalog.tableExists("business_to_business.gold.orders")):
    df_gold.write\
    .format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema", True)\
    .saveAsTable("business_to_business.gold.fact_sborders")
else:
    gold_delta = DeltaTable.forName(spark, "business_to_business.gold.fact_sborders")
    gold_delta.alias("gold").merge(
        source = df_gold.alias("s"),
        condition= "gold.order_id = s.order_id AND gold.date = s.date AND gold.customer_code = s.customer_code AND gold.product_code = s.product_code"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


### Merging with the Parent Company

In [0]:
child_df = spark.sql("select date, product_code, customer_code, sold_quantity from business_to_business.gold.fact_sborders")
display(child_df.limit(10))

In [0]:
from pyspark.sql import functions as F

In [0]:
child_df = child_df.withColumn("month_start",F.trunc("date", "MM"))\
    .groupBy("customer_code", "product_code","month_start")\
    .agg(F.sum("sold_quantity").alias("sold_quantity"))\
        .select("product_code", "month_start", "customer_code", "sold_quantity")


In [0]:
child_df.show(3)

In [0]:
child_df.show()


In [0]:
child_df.printSchema()

In [0]:
child_df = child_df.withColumnRenamed("month_start", "date")

In [0]:
child_df.show()

In [0]:
from delta.tables import DeltaTable
gold_parent_delta = DeltaTable.forName(spark, "business_to_business.gold.fact_orders")
gold_parent_delta.alias("parent").merge(
    source=child_df.alias("child"),
    condition="parent.product_code = child.product_code AND parent.date = child.date AND parent.customer_code = child.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()